# Module 1: Data Pipeline — Zepto Data & AI Platform
This notebook walks through the raw-to-relational data pipeline for catalog benchmarking:
1. **Scraping** catalog products across $\ge 3$ categories from `books.toscrape.com`
2. **Cleaning & Type Parsing** (`price_gbp`, `rating`, `in_stock`)
3. **Fixed-rate currency conversion** ($1\text{ GBP} = 105.50\text{ INR}$)
4. **Normalized Relational Ingestion** into SQLite (`categories` and `books` tables with PK/FK)
5. **SQL Analytics Queries** demonstrating required clauses (`SELECT/WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN/BETWEEN`, `JOIN`)
6. **Side-by-side Equivalence** of `pd.read_sql` and `pd.merge`

In [ ]:
import os
import sqlite3
import pandas as pd
from scraper import (
    fetch_exchange_rate_with_fallback,
    scrape_all_categories,
    clean_book_data,
    FIXED_GBP_TO_INR_RATE
)
from db_loader import (
    init_database,
    load_data_to_db,
    run_sql_queries,
    verify_sql_vs_pandas_merge
)

## Step 1: Fixed Currency Conversion Baseline
Required rate: **1 GBP = 105.50 INR** (fixed project constant).

In [ ]:
conversion_rate = fetch_exchange_rate_with_fallback(FIXED_GBP_TO_INR_RATE)
print(f"Active Project Conversion Baseline: 1 GBP = {conversion_rate} INR")

## Step 2: Web Scraping Catalog Data
We scrape across 5 categories (Mystery, Historical Fiction, Travel, Classics, Philosophy) to ensure $\ge 60$ books across $\ge 3$ categories.

In [ ]:
raw_books = scrape_all_categories()
print(f"Total raw items scraped: {len(raw_books)}")

## Step 3: Data Cleaning & Type Conversion

In [ ]:
df_cleaned = clean_book_data(raw_books, conversion_rate=conversion_rate)
print(f"Dataset shape: {df_cleaned.shape}")
df_cleaned.head(10)

## Step 4: Loading into Normalized SQLite Schema
Loads into `categories` and `books` tables with foreign keys enabled.

In [ ]:
db_path = "zepto_catalog.db"
load_data_to_db(df_cleaned, db_path=db_path)
print("Database populated successfully.")

## Step 5: Executing SQL Queries
We execute 6 queries covering SELECT/WHERE, ORDER BY, LIMIT, DISTINCT, BETWEEN/IN, and JOIN.

In [ ]:
query_results = run_sql_queries(db_path=db_path)

## Step 6: Side-by-side Equivalence: `pd.read_sql` vs. `pd.merge`

In [ ]:
df_sql, df_merge, is_equiv = verify_sql_vs_pandas_merge(db_path=db_path)
print(f"Strict Mathematical Equivalence Verified: {is_equiv}")